# Agent training

In [ ]:
from training_utils import train_model, further_train_model, evaluate_model

def compare_all_models(algo_dict):
    """
    algo_dict = {
        "RANDOM": None,
        "DQN": "runs/v0.7.4_circle_DQN_2024-12-10_13-42-17.zip"
    }
    """
    for algo, model in algo_dict.items():
        evaluate_model(
            algorithm=algo,
            version_tag="v0.7.5",
            reward_strategy="basic",
            map="circle",
            n_vehicles=15,
            n_episodes= 50,
            render_mode=None,
            model_load_path=model,
            random_seed=123
        )

def make_model_path(run_dir, model_id=None): 
    if model_id is None:
        model_id = run_dir
    return f"runs/{run_dir}/{model_id}.zip"

# ----- Visualisations ----

import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import json
from datetime import datetime
import os

def make_violinplot(data_df, metric_name, save_path=None):
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(10, 6))    # Create plots
    sns.set_theme(style="whitegrid")

    plot = sns.violinplot(x="algorithm", y="Value", data=data_df, palette="muted", cut = 0)
    # add in for smaller datasets:
    # sns.stripplot(df_filtered, x="algorithm", y="Value", color=".3")

    plt.title(f"Comparison of {metric_name}", fontsize=14)
    plt.xlabel("Algorithm")
    plt.ylabel("Value")
    plt.grid(axis="y", linestyle="-", alpha=0.7) 

    sns.despine(left=True, bottom=True)
    plt.show()

    if save_path:
        fig = plot.get_figure()
        fig.savefig(save_path)
        

def plot_results(filepath_list, metrics_to_plot="all", save_figure=False):
    data_list = []
    for filepath in filepath_list:
        with open(filepath) as file:
            data_list.extend(json.load(file))

    data = data_list
    df = pd.DataFrame(data)

    # Select relevant numerical metrics for plotting. If "all" was specified instead of a list, overwrite it with all available metrics.
    if metrics_to_plot == "all":
        metrics_to_plot = [
            "env/charging_stops_per_episode_mean",
            "env/global_ttt",
            "env/global_ttt_only_terminated",
            "env/ttt_per_ev_mean",
            "env/ttt_per_ev_mean_only_terminated",
            "env/cumulated_waiting_time",
            "env/cumulated_waiting_time_only_terminated",
            "env/empty_vehicles_per_episode",
            "env/final_simulation_time",
        ]

    # Melt the DataFrame to long format
    df_melted = df.melt(id_vars=["algorithm"], 
                        value_vars=metrics_to_plot, 
                        var_name="Metric", 
                        value_name="Value")

    if save_figure:
        # Save dataframe for easier later modifications of visuals
        save_directory = f"./visuals/{datetime.now()}"
        os.makedirs(save_directory, exist_ok=True)
        df.to_csv(f"{save_directory}/data_raw.csv", index=False)
        df_melted.to_csv(f"{save_directory}/data_melted.csv", index=False)


    # Create plots
    for metric in metrics_to_plot:
        # Filter data for the current metric
        df_filtered = df_melted[df_melted["Metric"] == metric]
        # Cut out the leading "/env" in the variable name
        metric_display_name = metric[4:]
        figure_save_path = f"{save_directory}/{metric_display_name}.png" if save_figure else None
        make_violinplot(data_df=df_filtered, metric_name=metric_display_name, save_path=figure_save_path)

def train_and_evaluate(algorithm, policy, version_tag, reward_strategy, map, n_vehicles, n_steps, n_nmevs=None, execution_context="local", random_seed_training=None, random_seed_eval=123, eval_episodes=50):
    """
    Returns:
        model_path (str): The file path of the trained model
        eval_metrics_filepath_list (List): A list of the filepaths for the evaluation metrics files of the trained model, the RANDOM and the GREEDY baseline algorithm
    """
    # train new model
    model_path = train_model(algorithm, policy, version_tag, reward_strategy, map, n_vehicles=n_vehicles, n_steps=n_steps, n_nmevs=n_nmevs, execution_context=execution_context, random_seed=random_seed_training)
    # evaluate with random and greedy
    model_evaluation_path = evaluate_model(algorithm, version_tag, reward_strategy, map, n_vehicles, n_nmevs=n_nmevs, n_episodes=eval_episodes, model_load_path=model_path, execution_context=execution_context, render_mode=None, random_seed=random_seed_eval)
    random_evaluation_path = evaluate_model("RANDOM", version_tag, reward_strategy, map, n_vehicles, n_nmevs=n_nmevs, n_episodes=eval_episodes, model_load_path=None, execution_context=execution_context, render_mode=None, random_seed=random_seed_eval)
    greedy_evaluation_path = evaluate_model("GREEDY", version_tag, reward_strategy, map, n_vehicles, n_nmevs=n_nmevs, n_episodes=eval_episodes, model_load_path=None, execution_context=execution_context, render_mode=None, random_seed=random_seed_eval)
    eval_metrics_filepath_list = [random_evaluation_path, greedy_evaluation_path, model_evaluation_path]
    return model_path, eval_metrics_filepath_list


In [ ]:
filepath_list = [
    "runs/2025-06-08_18-27-57_v0.7.9_basic_straight100Test_RANDOM/evaluation/metrics2025-06-08_18-27-56.json",
    "runs/2025-06-08_18-30-22_v0.7.9_basic_straight100Test_GREEDY/evaluation/metrics2025-06-08_18-30-21.json",
    "runs/2025-06-08_18-07-18_v0.7.9_basic_straight100Test_PPO/evaluation/metrics2025-06-08_18-25-25.json",
]
metrics_to_plot = [
    "env/ttt_per_ev_mean",
    "env/cumulated_waiting_time",
    "env/empty_vehicles_per_episode",
]
plot_results(filepath_list, metrics_to_plot, save_figure=True)


In [ ]:
# filepath_list = [
#     "runs/2025-04-04_12-24-57_v0.7.5_basic_circleSameSOC_PPO/evaluation/metrics2025-04-04_13-03-37.json",
#     "runs/2025-04-04_13-04-54_v0.7.5_basic_circleSameSOC_RANDOM/evaluation/metrics2025-04-04_13-04-53.json",
#     "runs/2025-04-04_13-06-12_v0.7.5_basic_circleSameSOC_GREEDY/evaluation/metrics2025-04-04_13-06-11.json"
# ]

model_path, filepath_list = train_and_evaluate("PPO", "MultiInputPolicy", "v0.7.9", "basic", "straight100Test_50MEV_150NMEV", n_vehicles=50, n_nmevs=150, n_steps=10000, execution_context="local")

In [ ]:
metrics_to_plot = [
    "env/ttt_per_ev_mean",
    "env/cumulated_waiting_time",
    "env/empty_vehicles_per_episode",
]
plot_results(filepath_list, metrics_to_plot, save_figure=True)

In [ ]:
algo_dict = {
    # "RANDOM": None,
    # "GREEDY": None,
    "A2C": make_model_path("2025-03-01_14-19-38_v0.7.4_basic_circle_A2C"),
    "A2C": make_model_path("2025-03-01_14-50-17_v0.7.4_shaping_circle_A2C"),
    "PPO": make_model_path("2025-03-01_13-33-24_v0.7.4_basic_circle_PPO"),
    "PPO": make_model_path("2025-03-01_12-42-26_v0.7.4_shaping_circle_PPO"),
    "DQN": make_model_path("2025-03-01_15-35-16_v0.7.4_basic_circle_DQN"),
    "DQN": make_model_path("2025-03-01_16-00-51_v0.7.4_shaping_circle_DQN"),
}

compare_all_models(algo_dict)

In [ ]:
evaluate_model(
    algorithm="GREEDY", 
    version_tag="v0.7.5", 
    reward_strategy="basic", 
    map="circleSameSOC", 
    n_vehicles=50, 
    n_episodes=50, 
    render_mode=None, 
    model_load_path=None, 
    random_seed=123
)

In [ ]:
evaluate_model(
    algorithm="PPO",
    version_tag="v0.7.5",
    reward_strategy="basic",
    map="circleSameSOC",
    n_vehicles=50,
    n_episodes= 50,
    render_mode=None,
    model_load_path=model_path,
    random_seed=123
)

In [ ]:
# Train new model
from training_utils import train_model

model_path = train_model(
    scenario="all_random",
    algorithm="PPO", 
    policy="MultiInputPolicy", 
    version_tag="v0.7.9", 
    reward_strategy="basic", 
    map="straight100Test", 
    n_vehicles=2, n_steps=1, execution_context="local", random_seed=None, use_wandb=True, wandb_entity="evcs-rl", wandb_project="v0.7.9_straight100Test_PPO")

In [ ]:
# Train new model
model_path = train_model(
    scenario="all_random",
    algorithm="DQN", 
    policy="MultiInputPolicy", 
    version_tag="v0.7.4", 
    reward_strategy="shaping", 
    map="circle", 
    n_vehicles=5, n_steps=50000, execution_context="local", random_seed=None)

In [ ]:
# manually configure model load path
run_dir = "v0.7.4_circle_DQN_2024-12-10_13-42-17"
model_id = "v0.7.4_circle_DQN_2024-12-10_13-42-17"
model_path = make_model_path(run_dir, model_id)

In [ ]:
# Train saved model further
model_path = further_train_model("PPO", "v0.7.5", "basic", "circleSameSOC", n_vehicles=5, n_steps=100000, model_load_path=model_path)

In [ ]:
# Quick evaluation
# model_path = "runs/v0.3_circle_a2c_2024-09-18_23-58-11/v0.3_circle_a2c_2024-09-18_23-58-11.zip"
evaluate_model("DQN", "v0.7.4", "noTime", "circle", n_vehicles=5, n_episodes=5, model_load_path=model_path)

In [ ]:
from stable_baselines3 import PPO, A2C, DQN
from environment import CircleEnv

# Observe execution of trained agent in GUI
env = CircleEnv(scenario_generator="all_random", render_mode="human", vehicles_to_spawn=5)
model = A2C.load(model_save_name, env)

num_steps = 5
observation, info = env.reset()
for t in range(num_steps):
        actions, _ = model.predict(observation, state=None, deterministic=False)
        observation, reward, terminated, truncated, info = env.step(actions)

env.close()

In [ ]:
# Random actions to compare with the agent
env = CircleEnv(scenario_generator="all_random", render_mode="human", vehicles_to_spawn=5)
observation, info = env.reset()
try:
    for _ in range(5):
        action = env.action_space.sample() # select a random action
        observation, reward, terminated, truncated, info = env.step(action)
        # if terminated or truncated:
            # observation, info = env.reset()
finally:        
    env.close()

### Benchmarks

In [ ]:
# --- Benchmark environment ---

def benchmark_env(random_seed):
    env = CircleEnv(scenario_generator="all_random", render_mode=None, vehicles_to_spawn=15)
    # set seed for reproducability
    import random
    random.seed(random_seed) # needed for batteries of simulation class TODO: make this seedable via the env.seed of gymnasium
    observation, info = env.reset(seed=random_seed)
    env.action_space.seed(random_seed)
    try:
        for _ in range(200):
            action = env.action_space.sample() # select a random action
            observation, reward, terminated, truncated, info = env.step(action)
            if terminated or truncated:
                observation, info = env.reset()
    finally:
        env.close()
    
import cProfile
# cProfile.run('train_model("A2C", "MultiInputPolicy", "v0.3", "circle", n_vehicles=5, n_steps=10000)', 'output.pstats')
benchmark_file = 'output_v0.3.1.pstats'
cProfile.run('benchmark_env(random_seed=1)', benchmark_file)

In [ ]:
import pstats
from pstats import SortKey
p = pstats.Stats(benchmark_file)
p.sort_stats(SortKey.CUMULATIVE).print_stats()